# 🔗 Dars 3: JOINlar va Subquerylar (SQLite)

**Modul:** 3 - Database va SQL  
**Dars:** 3  
**Mavzu:** SQL JOIN operatorlari va ichki so'rovlar (Subqueries)  

---

## 📖 Dars haqida

Bu darsda biz SQL ning eng muhim va kuchli xususiyatlaridan biri - **JOIN operatorlari** va **subquerylar** (ichki so'rovlar) bilan tanishamiz. Bu vositalar bir nechta jadvaldan ma'lumotlarni birlashtirish va murakkab so'rovlar yaratish imkonini beradi.

### 🎯 O'quv maqsadlari

Bu darsdan keyin siz:
- ✅ Barcha JOIN turlarini (INNER, LEFT, RIGHT, FULL, CROSS, SELF) qo'lla oladi
- ✅ Murakkab subquerylar yozishni o'rganadi
- ✅ Correlated subqueries bilan ishlashni biladi
- ✅ EXISTS va NOT EXISTS operatorlarini ishlatadi
- ✅ Real loyihalarda JOIN va subquerylardan foydalana oladi

---

## 🔗 SQLite ga ulanish

Ma'lumotlar bazasiga ulanish uchun quyidagi kodni ishga tushiramiz:

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns

# SQLite ulanish parametrlari
DB_PATH = 'datasets/lesson_joins.db'

# Ulanish yaratish
try:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # SQLAlchemy engine yaratish
    engine = create_engine(f"sqlite:///{DB_PATH}")
    
except Exception as e:
    print(f"❌ Ulanishda xatolik: {e}")

## 📊 Ma'lumotlar bazasi tuzilishi

Bizning ma'lumotlar bazasida quyidagi jadvallar mavjud:

1. **regions** - Viloyatlar
2. **categories** - Kategoriyalar  
3. **suppliers** - Yetkazib beruvchilar
4. **customers** - Mijozlar
5. **departments** - Bo'limlar
6. **employees** - Xodimlar
7. **products** - Mahsulotlar
8. **orders** - Buyurtmalar
9. **order_details** - Buyurtma tafsilotlari
10. **reviews** - Sharhlar

Keling, har bir jadvalning tuzilishini ko'rib chiqamiz:

In [4]:
def show_tables():
    query = """
    SELECT name as table_name
    FROM sqlite_master
    WHERE type = 'table'
    AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
    """
    
    df = pd.read_sql(query, engine)
    print("📋 Mavjud jadvallar:")
    for table in df['table_name']:
        print(f"  • {table}")
    
    return df

# Jadvallar ro'yxatini ko'rsatish
tables_df = show_tables()

📋 Mavjud jadvallar:
  • categories
  • customers
  • departments
  • employees
  • order_details
  • orders
  • products
  • regions
  • reviews
  • suppliers


## 1️⃣ INNER JOIN - Ichki birlashtirish

### 📖 Nazariya

**INNER JOIN** - faqat ikkala jadvalda ham mos kelgan yozuvlarni qaytaradi. Agar biror jadvalda mos keluvchi yozuv bo'lmasa, u yozuv natijada ko'rinmaydi.

**Sintaksis:**
```sql
SELECT columns
FROM table1
INNER JOIN table2 ON table1.column = table2.column;
```

### 🔍 Visual ko'rinish

```
Table A    Table B    INNER JOIN
┌───┐     ┌───┐     ┌───┐
│ 1 │ ──► │ 1 │ ──► │ 1 │
├───┤     ├───┤     ├───┤
│ 2 │     │ 2 │ ──► │ 2 │
├───┤     ├───┤     ├───┤
│ 3 │     │ 4 │     │ 3 │
└───┘     └───┘     └───┘
```

### 💡 Misollar

In [3]:
# 1-misol: Mijozlar va ularning buyurtmalari
query1 = """
SELECT 
    c.customer_id,
    c.first_name || ' ' || c.last_name as full_name,
    c.email,
    o.order_id,
    o.order_date,
    o.total_amount
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_id, o.order_date
LIMIT 10;
"""

print("📊 1-misol: Mijozlar va ularning buyurtmalari (INNER JOIN)")
df1 = pd.read_sql(query1, engine)
df1

📊 1-misol: Mijozlar va ularning buyurtmalari (INNER JOIN)


,customer_id,full_name,email,order_id,order_date,total_amount
0,1,Christopher Mills,qismoilov@example.net,58,2025-01-05,4.785479e+07
1,1,Christopher Mills,qismoilov@example.net,736,2025-07-10,1.218782e+08
2,2,Javohir Durdonova,samandarovmiran@example.org,923,2025-02-06,3.590992e+07
3,3,Osiyo Gallagher,michaelfields@example.net,6,2025-03-11,1.030550e+08
4,4,Osiyo Wright,rochavictor@example.net,1028,2024-11-27,2.989983e+07
5,4,Osiyo Wright,rochavictor@example.net,29,2024-12-15,5.715598e+07
6,4,Osiyo Wright,rochavictor@example.net,1338,2025-06-04,5.793117e+07
7,5,Tamara Mcclure,mmahmudov@example.com,1780,2025-02-06,1.593731e+08
8,6,Michael Osiyeva,neverett@example.net,800,2024-12-19,6.491825e+07
9,6,Michael Osiyeva,neverett@example.net,985,2025-07-17,1.078104e+08


In [ ]:
# 2-misol: Mahsulotlar va ularning kategoriyalari
query2 = """
SELECT 
    p.product_name,
    p.unit_price,
    c.category_name,
    s.company_name as supplier
FROM products p
INNER JOIN categories c ON p.category_id = c.category_id
INNER JOIN suppliers s ON p.supplier_id = s.supplier_id
WHERE p.is_active = 1
ORDER BY c.category_name, p.product_name
LIMIT 15;
"""

print("📦 2-misol: Mahsulotlar va ularning kategoriyalari (Multiple INNER JOINs)")
df2 = pd.read_sql(query2, engine)
df2

In [ ]:
# 3-misol: Xodimlar va ularning bo'limlari
query3 = """
SELECT 
    e.first_name || ' ' || e.last_name as employee_name,
    e.position_title,
    e.salary,
    d.department_name,
    r.region_name
FROM employees e
INNER JOIN departments d ON e.department_id = d.department_id
INNER JOIN regions r ON e.region_id = r.region_id
WHERE e.is_active = 1
ORDER BY d.department_name, e.salary DESC
LIMIT 10;
"""

print("👥 3-misol: Xodimlar va ularning bo'limlari")
df3 = pd.read_sql(query3, engine)
df3

## 2️⃣ LEFT JOIN - Chap tashqi birlashtirish

### 📖 Nazariya

**LEFT JOIN** (yoki LEFT OUTER JOIN) - chap jadvaldagi barcha yozuvlarni qaytaradi va o'ng jadvaldan mos kelgan yozuvlarni qo'shadi. Agar o'ng jadvalda mos keluvchi yozuv bo'lmasa, NULL qiymatlar qaytariladi.

**Sintaksis:**
```sql
SELECT columns
FROM table1
LEFT JOIN table2 ON table1.column = table2.column;
```

### 🔍 Visual ko'rinish

```
Table A    Table B    LEFT JOIN
┌───┐     ┌───┐     ┌───┐
│ 1 │ ──► │ 1 │ ──► │ 1 │
├───┤     ├───┤     ├───┤
│ 2 │     │ 2 │ ──► │ 2 │
├───┤     ├───┤     ├───┤
│ 3 │     │   │     │ 3 │ NULL
└───┘     └───┘     └───┘
```

### 💡 Misollar

In [ ]:
# 1-misol: Barcha mijozlar va ularning buyurtmalari (buyurtmasi bo'lmaganlar ham)
query4 = """
SELECT 
    c.customer_id,
    c.first_name || ' ' || c.last_name as full_name,
    c.customer_type,
    COUNT(o.order_id) as total_orders,
    COALESCE(SUM(o.total_amount), 0) as total_spent,
    MAX(o.order_date) as last_order_date
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.customer_type
ORDER BY total_orders DESC, total_spent DESC
LIMIT 15;
"""

print("🛒 1-misol: Barcha mijozlar va ularning buyurtmalari (LEFT JOIN)")
df4 = pd.read_sql(query4, engine)
df4

In [ ]:
# 2-misol: Barcha mahsulotlar va ularning sharhlari
query5 = """
SELECT 
    p.product_name,
    p.unit_price,
    COUNT(r.review_id) as review_count,
    ROUND(AVG(r.rating), 2) as avg_rating,
    CASE 
        WHEN COUNT(r.review_id) = 0 THEN 'Sharhsiz'
        WHEN AVG(r.rating) >= 4.5 THEN 'Ajoyib'
        WHEN AVG(r.rating) >= 4.0 THEN 'Yaxshi'
        WHEN AVG(r.rating) >= 3.0 THEN 'O\'rtacha'
        ELSE 'Past'
    END as rating_category
FROM products p
LEFT JOIN reviews r ON p.product_id = r.product_id
WHERE p.is_active = true
GROUP BY p.product_id, p.product_name, p.unit_price
ORDER BY review_count DESC, avg_rating DESC
LIMIT 10;
"""

print("⭐ 2-misol: Barcha mahsulotlar va ularning sharhlari")
df5 = pd.read_sql(query5, engine)
df5

## 3️⃣ RIGHT JOIN - O'ng tashqi birlashtirish

### 📖 Nazariya

**RIGHT JOIN** (yoki RIGHT OUTER JOIN) - o'ng jadvaldagi barcha yozuvlarni qaytaradi va chap jadvaldan mos kelgan yozuvlarni qo'shadi. LEFT JOIN ning teskarisi.

**Sintaksis:**
```sql
SELECT columns
FROM table1
RIGHT JOIN table2 ON table1.column = table2.column;
```

### 💡 Misollar

In [ ]:
# 1-misol: Barcha bo'limlar va ulardagi xodimlar
query6 = """
SELECT 
    d.department_name,
    d.budget,
    COUNT(e.employee_id) as employee_count,
    ROUND(AVG(e.salary), 2) as avg_salary,
    MIN(e.salary) as min_salary,
    MAX(e.salary) as max_salary
FROM departments d
RIGHT JOIN employees e ON d.department_id = e.department_id
WHERE e.is_active = true
GROUP BY d.department_id, d.department_name, d.budget
ORDER BY employee_count DESC;
"""

print("🏢 1-misol: Barcha bo'limlar va ulardagi xodimlar (RIGHT JOIN)")
df6 = pd.read_sql(query6, engine)
df6

## 4️⃣ FULL OUTER JOIN - To'liq tashqi birlashtirish

### 📖 Nazariya

**FULL OUTER JOIN** - ikkala jadvaldagi barcha yozuvlarni qaytaradi. Mos kelgan yozuvlar birlashtiriladi, mos kelmaydiganlar uchun NULL qiymatlar qaytariladi.

**Sintaksis:**
```sql
SELECT columns
FROM table1
FULL OUTER JOIN table2 ON table1.column = table2.column;
```

### 🔍 Visual ko'rinish

```
Table A    Table B    FULL OUTER JOIN
┌───┐     ┌───┐     ┌───┐
│ 1 │ ──► │ 1 │ ──► │ 1 │
├───┤     ├───┤     ├───┤
│ 2 │     │ 2 │ ──► │ 2 │
├───┤     ├───┤     ├───┤
│ 3 │     │   │     │ 3 │ NULL
├───┤     ├───┤     ├───┤
│   │     │ 4 │     │NULL│ 4
└───┘     └───┘     └───┘
```

### 💡 Misollar

In [ ]:
# 1-misol: Barcha mijozlar va barcha buyurtmalar (FULL OUTER JOIN)
query7 = """
SELECT 
    COALESCE(c.first_name || ' ' || c.last_name, 'NOMA''LUM') as customer_name,
    COALESCE(o.order_number, 'BUYURTMA YO''Q') as order_number,
    o.order_date,
    o.total_amount,
    CASE 
        WHEN c.customer_id IS NOT NULL AND o.order_id IS NOT NULL THEN 'Mos kelgan'
        WHEN c.customer_id IS NOT NULL AND o.order_id IS NULL THEN 'Faqat mijoz'
        WHEN c.customer_id IS NULL AND o.order_id IS NOT NULL THEN 'Faqat buyurtma'
    END as join_type
FROM customers c
FULL OUTER JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_id, o.order_date
LIMIT 20;
"""

print("🔄 1-misol: Barcha mijozlar va barcha buyurtmalar (FULL OUTER JOIN)")
df7 = pd.read_sql(query7, engine)
df7

## 5️⃣ CROSS JOIN - Kartezian ko'paytirish

### 📖 Nazariya

**CROSS JOIN** - ikkala jadvaldagi barcha yozuvlarning barcha kombinatsiyalarini qaytaradi. Bu juda katta natija berishi mumkin!

**Sintaksis:**
```sql
SELECT columns
FROM table1
CROSS JOIN table2;
```

### ⚠️ Ogohlantirish
CROSS JOIN dan ehtiyot bo'ling! Agar table1 da 100 ta yozuv, table2 da 100 ta yozuv bo'lsa, natija 10,000 ta yozuv bo'ladi.

### 💡 Misollar

In [ ]:
# 1-misol: Barcha kategoriyalar va barcha yetkazib beruvchilar kombinatsiyasi
query8 = """
SELECT 
    c.category_name,
    s.company_name,
    r.region_name as supplier_region
FROM categories c
CROSS JOIN suppliers s
LEFT JOIN regions r ON s.region_id = r.region_id
WHERE c.parent_category_id IS NULL  -- Faqat asosiy kategoriyalar
ORDER BY c.category_name, s.company_name
LIMIT 20;
"""

print("🔀 1-misol: Kategoriyalar va yetkazib beruvchilar kombinatsiyasi (CROSS JOIN)")
df8 = pd.read_sql(query8, engine)
df8

## 6️⃣ SELF JOIN - Jadvalning o'zi bilan birlashtirish

### 📖 Nazariya

**SELF JOIN** - jadvalning o'zini o'zi bilan birlashtirish. Bu iyerarxik ma'lumotlar (masalan, menejer-xodim munosabatlari) uchun juda foydali.

**Sintaksis:**
```sql
SELECT columns
FROM table1 t1
JOIN table1 t2 ON t1.column = t2.column;
```

### 💡 Misollar

In [ ]:
# 1-misol: Xodimlar va ularning menejerlari
query9 = """
SELECT 
    e.first_name || ' ' || e.last_name as employee_name,
    e.position_title as employee_position,
    e.salary as employee_salary,
    m.first_name || ' ' || m.last_name as manager_name,
    m.position_title as manager_position,
    m.salary as manager_salary,
    (m.salary - e.salary) as salary_difference
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.employee_id
WHERE e.is_active = true
ORDER BY m.employee_id, e.salary DESC
LIMIT 15;
"""

print("👔 1-misol: Xodimlar va ularning menejerlari (SELF JOIN)")
df9 = pd.read_sql(query9, engine)
df9

In [ ]:
# 2-misol: Kategoriyalar iyerarxiyasi
query10 = """
SELECT 
    c.category_name as subcategory,
    p.category_name as parent_category,
    c.description as subcategory_desc,
    p.description as parent_desc
FROM categories c
LEFT JOIN categories p ON c.parent_category_id = p.category_id
ORDER BY p.category_name, c.category_name;
"""

print("📂 2-misol: Kategoriyalar iyerarxiyasi (SELF JOIN)")
df10 = pd.read_sql(query10, engine)
df10

## 🔍 Subquerylar (Ichki so'rovlar)

### 📖 Nazariya

**Subquery** (ichki so'rov) - boshqa SQL so'rovi ichida joylashgan SQL so'rovi. Subquerylar quyidagi joylarda ishlatilishi mumkin:

1. **SELECT** bandida
2. **FROM** bandida  
3. **WHERE** bandida
4. **HAVING** bandida

### 📝 Turlari:
- **Oddiy subquery** - tashqi so'rovdan mustaqil
- **Correlated subquery** - tashqi so'rovga bog'liq
- **Scalar subquery** - bitta qiymat qaytaradi
- **Multi-row subquery** - bir nechta qator qaytaradi

---

## 7️⃣ WHERE bandida subquery

### 💡 Misollar

In [ ]:
# 1-misol: O'rtacha narxdan qimmat mahsulotlar
query11 = """
SELECT 
    product_name,
    unit_price,
    category_id
FROM products
WHERE unit_price > (
    SELECT AVG(unit_price) 
    FROM products 
    WHERE is_active = true
)
AND is_active = true
ORDER BY unit_price DESC
LIMIT 10;
"""

print("💰 1-misol: O'rtacha narxdan qimmat mahsulotlar (Subquery in WHERE)")
df11 = pd.read_sql(query11, engine)
df11

In [ ]:
# 2-misol: Eng ko'p buyurtma bergan mijozlar
query12 = """
SELECT 
    first_name || ' ' || last_name as customer_name,
    email,
    customer_type,
    total_spent
FROM customers
WHERE total_spent > (
    SELECT AVG(total_spent) 
    FROM customers 
    WHERE total_spent > 0
)
ORDER BY total_spent DESC
LIMIT 10;
"""

print("🏆 2-misol: Eng ko'p xarid qilgan mijozlar")
df12 = pd.read_sql(query12, engine)
df12

In [ ]:
# 3-misol: IN operatori bilan subquery
query13 = """
SELECT 
    first_name || ' ' || last_name as customer_name,
    email,
    region_id
FROM customers
WHERE region_id IN (
    SELECT region_id 
    FROM regions 
    WHERE population > 2000000
)
ORDER BY region_id, customer_name
LIMIT 15;
"""

print("🏙️ 3-misol: Katta shaharlardagi mijozlar (IN subquery)")
df13 = pd.read_sql(query13, engine)
df13

## 8️⃣ FROM bandida subquery

### 💡 Misollar

In [ ]:
# 1-misol: Har bir kategoriyadagi o'rtacha narx
query14 = """
SELECT 
    category_name,
    product_count,
    avg_price,
    min_price,
    max_price
FROM (
    SELECT 
        c.category_name,
        COUNT(p.product_id) as product_count,
        ROUND(AVG(p.unit_price), 2) as avg_price,
        MIN(p.unit_price) as min_price,
        MAX(p.unit_price) as max_price
    FROM categories c
    LEFT JOIN products p ON c.category_id = p.category_id
    WHERE p.is_active = true
    GROUP BY c.category_id, c.category_name
) category_stats
ORDER BY avg_price DESC;
"""

print("📊 1-misol: Kategoriyalar bo'yicha statistika (Subquery in FROM)")
df14 = pd.read_sql(query14, engine)
df14

In [ ]:
# 2-misol: Eng yaxshi xodimlar
query15 = """
SELECT 
    employee_name,
    department_name,
    total_sales,
    sales_rank
FROM (
    SELECT 
        e.first_name || ' ' || e.last_name as employee_name,
        d.department_name,
        COALESCE(SUM(o.total_amount), 0) as total_sales,
        RANK() OVER (ORDER BY COALESCE(SUM(o.total_amount), 0) DESC) as sales_rank
    FROM employees e
    LEFT JOIN departments d ON e.department_id = d.department_id
    LEFT JOIN orders o ON e.employee_id = o.employee_id
    WHERE e.is_active = true
    GROUP BY e.employee_id, e.first_name, e.last_name, d.department_name
) employee_sales
WHERE sales_rank <= 10
ORDER BY sales_rank;
"""

print("🥇 2-misol: Eng yaxshi xodimlar (Subquery with Window Function)")
df15 = pd.read_sql(query15, engine)
df15

## 9️⃣ Correlated Subqueries - Bog'liq ichki so'rovlar

### 📖 Nazariya

**Correlated subquery** - tashqi so'rovdagi qiymatlarga bog'liq bo'lgan ichki so'rov. Har bir qator uchun alohida bajariladi.

### 💡 Misollar

In [ ]:
# 1-misol: O'z bo'limidagi o'rtacha maoshdan yuqori maosh oluvchi xodimlar
query16 = """
SELECT 
    e.first_name || ' ' || e.last_name as employee_name,
    d.department_name,
    e.salary,
    dept_avg_salary,
    (e.salary - dept_avg_salary) as difference
FROM employees e
JOIN departments d ON e.department_id = d.department_id
JOIN (
    SELECT 
        department_id,
        ROUND(AVG(salary), 2) as dept_avg_salary
    FROM employees
    WHERE is_active = true
    GROUP BY department_id
) dept_stats ON e.department_id = dept_stats.department_id
WHERE e.salary > dept_stats.dept_avg_salary
AND e.is_active = true
ORDER BY difference DESC
LIMIT 10;
"""

print("💼 1-misol: O'z bo'limidagi o'rtacha maoshdan yuqori maosh oluvchi xodimlar")
df16 = pd.read_sql(query16, engine)
df16

In [ ]:
# 2-misol: Har bir mijozning oxirgi buyurtmasi
query17 = """
SELECT 
    c.first_name || ' ' || c.last_name as customer_name,
    c.email,
    o.order_date,
    o.total_amount,
    o.order_status
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_date = (
    SELECT MAX(order_date)
    FROM orders
    WHERE customer_id = c.customer_id
)
ORDER BY o.order_date DESC
LIMIT 15;
"""

print("🛍️ 2-misol: Har bir mijozning oxirgi buyurtmasi (Correlated subquery)")
df17 = pd.read_sql(query17, engine)
df17

## 1️⃣0️⃣ EXISTS va NOT EXISTS operatorlari

### 📖 Nazariya

**EXISTS** - ichki so'rov hech bo'lmaganda bitta qator qaytarsa TRUE qaytaradi
**NOT EXISTS** - ichki so'rov hech qator qaytarmasa TRUE qaytaradi

### 💡 Misollar

In [ ]:
# 1-misol: Buyurtma bergan mijozlar
query18 = """
SELECT 
    customer_id,
    first_name || ' ' || last_name as customer_name,
    email,
    customer_type,
    registration_date
FROM customers c
WHERE EXISTS (
    SELECT 1 
    FROM orders o 
    WHERE o.customer_id = c.customer_id
)
ORDER BY registration_date
LIMIT 10;
"""

print("✅ 1-misol: Buyurtma bergan mijozlar (EXISTS)")
df18 = pd.read_sql(query18, engine)
df18

In [ ]:
# 2-misol: Buyurtmasi bo'lmagan mijozlar
query19 = """
SELECT 
    customer_id,
    first_name || ' ' || last_name as customer_name,
    email,
    customer_type,
    registration_date
FROM customers c
WHERE NOT EXISTS (
    SELECT 1 
    FROM orders o 
    WHERE o.customer_id = c.customer_id
)
ORDER BY registration_date DESC
LIMIT 10;
"""

print("❌ 2-misol: Buyurtmasi bo'lmagan mijozlar (NOT EXISTS)")
df19 = pd.read_sql(query19, engine)
df19

In [ ]:
# 3-misol: Sharhi bor mahsulotlar
query20 = """
SELECT 
    product_name,
    unit_price,
    rating,
    reviews_count
FROM products p
WHERE EXISTS (
    SELECT 1 
    FROM reviews r 
    WHERE r.product_id = p.product_id 
    AND r.rating >= 4
)
AND is_active = true
ORDER BY rating DESC, reviews_count DESC
LIMIT 10;
"""

print("⭐ 3-misol: Yaxshi sharhga ega mahsulotlar (EXISTS)")
df20 = pd.read_sql(query20, engine)
df20

## 📊 Murakkab misollar

### 💡 1. Hisobot: Viloyatlar bo'yicha savdo tahlili

In [ ]:
# Murakkab hisobot: Viloyatlar bo'yicha savdo tahlili
query21 = """
SELECT 
    r.region_name,
    r.population,
    COUNT(DISTINCT c.customer_id) as total_customers,
    COUNT(DISTINCT o.order_id) as total_orders,
    ROUND(SUM(o.total_amount), 2) as total_revenue,
    ROUND(AVG(o.total_amount), 2) as avg_order_value,
    COUNT(DISTINCT p.product_id) as products_sold,
    ROUND(SUM(o.total_amount) / NULLIF(r.population, 0) * 1000000, 2) as revenue_per_million_people
FROM regions r
LEFT JOIN customers c ON r.region_id = c.region_id
LEFT JOIN orders o ON c.customer_id = o.customer_id
LEFT JOIN order_details od ON o.order_id = od.order_id
LEFT JOIN products p ON od.product_id = p.product_id
GROUP BY r.region_id, r.region_name, r.population
ORDER BY total_revenue DESC;
"""

print("📈 Murakkab hisobot: Viloyatlar bo'yicha savdo tahlili")
df21 = pd.read_sql(query21, engine)
df21

### 💡 2. Top mahsulotlar tahlili

In [ ]:
# Top mahsulotlar tahlili
query22 = """
SELECT 
    p.product_name,
    c.category_name,
    s.company_name as supplier,
    p.unit_price,
    sales.total_quantity,
    sales.total_revenue,
    sales.customer_count,
    ROUND(sales.total_revenue / NULLIF(sales.total_quantity, 0), 2) as avg_price_sold,
    p.rating,
    p.reviews_count
FROM products p
JOIN categories c ON p.category_id = c.category_id
JOIN suppliers s ON p.supplier_id = s.supplier_id
JOIN (
    SELECT 
        product_id,
        SUM(quantity) as total_quantity,
        SUM(line_total) as total_revenue,
        COUNT(DISTINCT order_id) as order_count,
        COUNT(DISTINCT customer_id) as customer_count
    FROM (
        SELECT 
            od.product_id,
            od.quantity,
            od.line_total,
            o.order_id,
            o.customer_id
        FROM order_details od
        JOIN orders o ON od.order_id = o.order_id
        WHERE o.order_status = 'Delivered'
    ) sales_data
    GROUP BY product_id
) sales ON p.product_id = sales.product_id
WHERE p.is_active = true
ORDER BY sales.total_revenue DESC
LIMIT 15;
"""

print("🏆 Top mahsulotlar tahlili")
df22 = pd.read_sql(query22, engine)
df22

## 🎯 Xulosa

Bu darsda biz quyidagi muhim tushunchalarni o'rgandik:

### 🔗 JOIN turlari:
1. **INNER JOIN** - Faqat mos kelgan yozuvlar
2. **LEFT JOIN** - Chap jadvaldagi barcha yozuvlar
3. **RIGHT JOIN** - O'ng jadvaldagi barcha yozuvlar
4. **FULL OUTER JOIN** - Barcha yozuvlar
5. **CROSS JOIN** - Barcha kombinatsiyalar
6. **SELF JOIN** - Jadvalning o'zi bilan birlashtirish

### 🔍 Subquery turlari:
1. **WHERE bandida** - Filtrlash uchun
2. **FROM bandida** - Virtual jadval sifatida
3. **Correlated subqueries** - Tashqi so'rovga bog'liq
4. **EXISTS/NOT EXISTS** - Mavjudlik tekshiruvi

### 💡 Qachon qaysi JOIN dan foydalanish:
- **INNER JOIN**: Faqat mos kelgan ma'lumotlar kerak bo'lganda
- **LEFT JOIN**: Chap jadvaldagi barcha ma'lumotlar kerak, o'ng jadvaldagilar ixtiyoriy
- **RIGHT JOIN**: O'ng jadvaldagi barcha ma'lumotlar kerak, chap jadvaldagilar ixtiyoriy
- **FULL OUTER JOIN**: Ikkala jadvaldagi barcha ma'lumotlar kerak
- **CROSS JOIN**: Barcha kombinatsiyalar kerak (kombinatorika)
- **SELF JOIN**: Iyerarxik ma'lumotlar (menejer-xodim, parent-child)

### 🚀 Keyingi qadamlar:
1. **Practical.ipynb** faylida amaliy mashqlar bajarish
2. **homework.ipynb** faylida uy vazifasini bajarish
3. Real loyihalarda JOIN va subquerylardan foydalanish

---

**🎉 Dars yakunlandi! Savollaringiz bormi?**

## 📚 Qo'shimcha resurslar

### 📖 O'quv materiallari:
- [PostgreSQL JOIN Documentation](https://www.postgresql.org/docs/current/queries-table-expressions.html)
- [SQL JOIN Visualizer](https://joins.spathon.com/)
- [W3Schools SQL JOINs](https://www.w3schools.com/sql/sql_join.asp)

### 🎥 Video materiallar:
- PostgreSQL JOIN tutorial
- Advanced SQL Subqueries
- Database Design & SQL Joins

### 📝 Amaliy mashqlar:
- `practical.ipynb` - Qo'shimcha mashqlar
- `homework.ipynb` - Uy vazifasi

---

*Bu material "Empowering Young Women in Data Science and AI" dasturi doirasida tayyorlangan*